# 09_register_toxicity_endpoints.py
**Register Toxicity Endpoints**

Register hERG, DILI, or other independent toxicity endpoint results.

This notebook is the one-to-one notebook version of `scripts/09_register_toxicity_endpoints.py`. The implementation below is copied from that script so the Python and notebook workflows stay aligned.

## 1. Project setup
This cell locates the repository root and makes `scripts/core` imports available.

In [1]:
from pathlib import Path
import os
import sys

# Make the notebook runnable whether Jupyter starts in the project root
# or directly inside the notebooks/ folder.
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)


Project root: /Users/himanshugoel/Downloads/claudecode/CompTox_Multimodal_Foundation_Model


## 2. Implementation

Every function below is copied verbatim from `scripts/09_register_toxicity_endpoints`, in source order, kept in sync by `python scripts/sync_notebooks.py` (and checked by `11_validate_project.py`). Edit the `.py` file, then re-run the sync script -- never hand-edit these cells.

In [ ]:
"""STEP 9 -- Append real, measured toxicity/injury endpoint results (hERG,
DILI, hepatotoxicity, etc.) into a single registry CSV, validating required
columns and identifiers first. Deliberately kept separate from *predicted*
targets (ToxProfiler) elsewhere in the project: a predicted KCNH2 target hit
must never be silently treated as a measured hERG result. This registry is
what 07_query_chemical.py and 08_build_knowledge_graph.py read as
"[TOXICITY / INJURY ENDPOINTS]"."""
import argparse
import pandas as pd
from core.paths import ENDPOINT_DATA

OUT=ENDPOINT_DATA/"endpoint_results.csv"; REQUIRED={"endpoint","result_type"}; IDENTIFIERS={"chemical_name","compound_id","DTXSID"}
ALL=["chemical_name","compound_id","DTXSID","endpoint","endpoint_group","result_type","task_type","value","label","probability","unit","model","source","notes"]


#### `main`

CLI entry point: validate `--input`'s columns/identifiers, then append

In [ ]:
def main():
    """CLI entry point: validate `--input`'s columns/identifiers, then append
    it to the registry (deduplicated) -- or with --check, just report the
    registry's current size."""
    p=argparse.ArgumentParser(); p.add_argument("--input"); p.add_argument("--check",action="store_true"); a=p.parse_args()
    if a.check:
        print("Endpoint registry:",OUT); print("Rows:",len(pd.read_csv(OUT)) if OUT.exists() else 0); return
    if not a.input: p.error("--input is required unless --check is used")
    new=pd.read_csv(a.input); missing=REQUIRED-set(new.columns)
    if missing: raise ValueError(f"Missing required columns: {sorted(missing)}")
    if not (IDENTIFIERS & set(new.columns)): raise ValueError("Provide at least one identifier: chemical_name, compound_id, or DTXSID")
    for col in ALL:
        if col not in new: new[col]=""
    new=new[ALL]; current=pd.read_csv(OUT) if OUT.exists() else pd.DataFrame(columns=ALL)
    merged=new.copy() if current.empty else pd.concat([current,new],ignore_index=True)
    merged=merged.drop_duplicates(); OUT.parent.mkdir(parents=True,exist_ok=True); merged.to_csv(OUT,index=False)
    print("Registered input rows:",len(new)); print("Unique endpoint rows:",len(merged)); print("Saved:",OUT)


In [ ]:
# Set RUN_STEP=True when you are ready to execute this workflow.
# The notebook defaults to False so "Run All" is safe and does not accidentally
# download large files, start a long training job, or overwrite project outputs.
RUN_STEP = False

# Command-line arguments used when RUN_STEP=True.
RUN_ARGS = ["--check"]

if RUN_STEP:
    old = sys.argv[:]
    try:
        sys.argv = ['09_register_toxicity_endpoints.py'] + RUN_ARGS
        try:
            main()
        except SystemExit as exc:
            # main() uses SystemExit(0) as a CLI success signal (e.g. --check).
            # A terminal treats that as silent success; Jupyter displays *any*
            # SystemExit as an error-looking traceback, so only re-raise on an
            # actual failure (nonzero/non-None exit code).
            if exc.code not in (0, None):
                raise
    finally:
        sys.argv = old
else:
    print('Implementation loaded successfully.')
    print("Set RUN_STEP = True in this cell to execute: 09_register_toxicity_endpoints.py")
    print('RUN_ARGS =', RUN_ARGS)
